# PourCastAI — Step 7: Silver + Bronze risk data -> Gold

Rebuilds gold_inventory_health and gold_shipments_open exactly as defined
in schema.sql, then adds gold_risk_scores -- the real 7-factor formula from
risk_agent.py's score_route(), computed here instead of per-chat-turn.

Weights (unchanged from risk_agent.py):
  weather_alert 25%  road_hazard 10%  precip_forecast 10%  distance 20%
  diesel 10%  reliability 15%  rural 10%

In [0]:
CATALOG = "pourcastai"
LANDING = f"/Volumes/{CATALOG}/bronze/landing"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

from pyspark.sql.functions import col, round as sround, when, max as smax, lit

### gold_inventory_health — latest snapshot only, same columns as schema.sql

In [0]:
fact_inv = spark.table(f"{CATALOG}.silver.fact_inventory_snapshot")
latest_date = fact_inv.agg(smax("snapshot_date")).collect()[0][0]

dim_store = spark.table(f"{CATALOG}.silver.dim_store")
dim_item = spark.table(f"{CATALOG}.silver.dim_item")

gold_inventory_health = (
    fact_inv.filter(col("snapshot_date") == latest_date)
    .join(dim_store.select("store_number", "store_name", "county"), "store_number")
    .join(dim_item.select("item_number", "item_description", "vendor_number",
                           "state_bottle_cost", "state_bottle_retail"), "item_number")
    .withColumn("inventory_value", sround(col("on_hand_bottles") * col("state_bottle_retail"), 2))
    .withColumn("low_stock_flag", when(col("on_hand_bottles") <= col("reorder_point"), 1).otherwise(0))
)

gold_inventory_health.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.gold.gold_inventory_health")
print(f"gold.gold_inventory_health: {gold_inventory_health.count()} rows "
      f"({gold_inventory_health.filter('low_stock_flag=1').count()} low-stock)")

gold.gold_inventory_health: 337 rows (65 low-stock)


### gold_shipments_open — same 4-table join as schema.sql

In [0]:
fact_shipments = spark.table(f"{CATALOG}.silver.fact_shipments")
dim_vendor = spark.table(f"{CATALOG}.silver.dim_vendor")
dim_carrier = spark.table(f"{CATALOG}.silver.dim_carrier")

gold_shipments_open = (
    fact_shipments
    .join(dim_store.select("store_number", "store_name", "population", "rural_flag", "county"), "store_number")
    .join(dim_item.select("item_number", "item_description", "state_bottle_retail"),
          "item_number")
    .join(dim_vendor.select("vendor_number", "vendor_name", "reliability_score"), "vendor_number")
    .join(dim_carrier.select("carrier_id", "carrier_name"), "carrier_id")
    .withColumn("shipment_value", sround(col("quantity_bottles") * col("state_bottle_retail"), 2))
)

gold_shipments_open.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.gold.gold_shipments_open")
print(f"gold.gold_shipments_open: {gold_shipments_open.count()} rows")
display(gold_shipments_open)

gold.gold_shipments_open: 8 rows


carrier_id,vendor_number,item_number,store_number,shipment_id,order_date,promised_eta,quantity_bottles,dest_lat,dest_long,origin_lat,origin_long,store_name,population,rural_flag,county,item_description,state_bottle_retail,vendor_name,reliability_score,carrier_name,shipment_value
1,260,43338,3773,2374,2026-08-10,2026-08-20,4,41.975787,-91.659795,41.699,-93.558,Benz Distributing,230000,0,LINN,Captain Morgan Spiced Rum,27.0,DIAGEO AMERICAS,0.878,Ruan,108.0
3,115,11777,3385,1588,2026-08-11,2026-08-19,23,42.031819,-91.67969,41.699,-93.558,Sam's Club 8162 / Cedar Rapids,230000,0,LINN,Black Velvet,9.95,CONSTELLATION BRANDS INC,0.854,Simulated Carrier B,228.85
2,260,11297,2633,323,2026-08-10,2026-08-20,114,41.554101,-93.596754,41.699,-93.558,Hy-Vee #3 / BDI / Des Moines,500000,0,POLK,Crown Royal Canadian Whisky,28.34,DIAGEO AMERICAS,0.878,Simulated Carrier A,3230.76
1,85,26827,3952,1285,2026-08-10,2026-08-20,39,41.529655,-90.48065,41.699,-93.558,Lot-A-Spirits,175000,0,SCOTT,Jack Daniels Old #7 Black Lbl,28.34,Brown Forman Corp.,0.934,Ruan,1105.26
3,260,43337,2512,1111,2026-08-10,2026-08-20,38,41.642764,-91.530463,41.699,-93.558,Hy-Vee Wine and Spirits / Iowa City,155000,0,JOHNSON,Captain Morgan Spiced Rum,17.63,DIAGEO AMERICAS,0.878,Simulated Carrier B,669.94
1,260,43338,2633,396,2026-08-10,2026-08-20,7,41.554101,-93.596754,41.699,-93.558,Hy-Vee #3 / BDI / Des Moines,500000,0,POLK,Captain Morgan Spiced Rum,27.0,DIAGEO AMERICAS,0.878,Ruan,189.0
2,260,11297,3952,1259,2026-08-10,2026-08-20,28,41.529655,-90.48065,41.699,-93.558,Lot-A-Spirits,175000,0,SCOTT,Crown Royal Canadian Whisky,28.34,DIAGEO AMERICAS,0.878,Simulated Carrier A,793.52
1,260,43337,2633,1312,2026-08-10,2026-08-20,207,41.554101,-93.596754,41.699,-93.558,Hy-Vee #3 / BDI / Des Moines,500000,0,POLK,Captain Morgan Spiced Rum,17.63,DIAGEO AMERICAS,0.878,Ruan,3649.41


### gold_risk_scores — the real formula, all 4 Bronze risk sources + diesel

In [0]:
osrm = spark.table(f"{CATALOG}.bronze.osrm_routes").select(
    "store_number", "distance_km", "duration_hr", col("is_live").alias("osrm_live"))
alerts = spark.table(f"{CATALOG}.bronze.nws_point_alerts").select(
    "store_number", col("severity").alias("weather_severity"),
    col("alert_count").alias("weather_alert_count"), col("is_live").alias("alerts_live"))
forecast = spark.table(f"{CATALOG}.bronze.weather_forecast").select(
    "store_number", "precip_prob", "wind_kph", col("is_live").alias("forecast_live"))
hazards_row = spark.table(f"{CATALOG}.bronze.nws_road_hazards").limit(1).collect()[0]
hazard_severity = hazards_row["severity"]

diesel_row = spark.read.csv(f"{LANDING}/diesel_price.csv", header=True, inferSchema=True).collect()[0]
diesel_price = float(diesel_row["price"])

risk_base = (
    gold_shipments_open
    .join(osrm, "store_number", "left")
    .join(alerts, "store_number", "left")
    .join(forecast, "store_number", "left")
)

# --- component risks, 0-1 each, matching score_route() exactly ---
forecast_severity = sround(
    (0.6 * col("precip_prob") + 0.4 * (col("wind_kph") / 50.0).cast("double")).cast("double"), 3)
forecast_severity = when(forecast_severity > 1.0, 1.0).otherwise(forecast_severity)

distance_risk = when(col("distance_km").isNotNull(),
                      (col("distance_km") / 300.0).cast("double")).otherwise(0.0)
distance_risk = when(distance_risk > 1.0, 1.0).otherwise(distance_risk)

diesel_risk = max(min((diesel_price - 3.50) / 1.50, 1.0), 0.0)   # scalar, same for every row

gold_risk_scores = risk_base \
    .withColumn("forecast_severity", forecast_severity) \
    .withColumn("distance_risk", distance_risk) \
    .withColumn("reliability_risk", 1.0 - col("reliability_score")) \
    .withColumn("rural_risk", when(col("rural_flag") == 1, 1.0).otherwise(0.0)) \
    .withColumn("risk_score", sround(100 * (
        0.25 * col("weather_severity") +
        0.10 * lit(hazard_severity) +
        0.10 * col("forecast_severity") +
        0.20 * col("distance_risk") +
        0.10 * lit(diesel_risk) +
        0.15 * col("reliability_risk") +
        0.10 * col("rural_risk")
    ), 1)) \
    .withColumn("risk_band", when(col("risk_score") >= 60, "HIGH")
                .when(col("risk_score") >= 30, "MEDIUM").otherwise("LOW")) \
    .withColumn("diesel_price_used", lit(diesel_price)) \
    .withColumn("road_hazard_severity", lit(hazard_severity))

In [0]:
gold_risk_scores.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.gold.gold_risk_scores")

print(f"gold.gold_risk_scores: {gold_risk_scores.count()} rows")
display(gold_risk_scores.select(
    "shipment_id", "store_name", "item_description", "risk_score", "risk_band",
    "distance_km", "weather_severity", "forecast_severity", "reliability_risk", "rural_risk"
).orderBy(col("risk_score").desc()))

gold.gold_risk_scores: 8 rows


shipment_id,store_name,item_description,risk_score,risk_band,distance_km,weather_severity,forecast_severity,reliability_risk,rural_risk
1259,Lot-A-Spirits,Crown Royal Canadian Whisky,23.9,LOW,272.4424,0.0,0.19,0.122,0.0
1285,Lot-A-Spirits,Jack Daniels Old #7 Black Lbl,23.1,LOW,272.4424,0.0,0.19,0.06599999999999995,0.0
1588,Sam's Club 8162 / Cedar Rapids,Black Velvet,17.6,LOW,180.01379999999997,0.0,0.139,0.14600000000000002,0.0
1111,Hy-Vee Wine and Spirits / Iowa City,Captain Morgan Spiced Rum,17.2,LOW,180.4613,0.0,0.135,0.122,0.0
2374,Benz Distributing,Captain Morgan Spiced Rum,16.8,LOW,176.14329999999998,0.0,0.127,0.122,0.0
323,Hy-Vee #3 / BDI / Des Moines,Crown Royal Canadian Whisky,6.6,LOW,19.7834,0.0,0.15,0.122,0.0
396,Hy-Vee #3 / BDI / Des Moines,Captain Morgan Spiced Rum,6.6,LOW,19.7834,0.0,0.15,0.122,0.0
1312,Hy-Vee #3 / BDI / Des Moines,Captain Morgan Spiced Rum,6.6,LOW,19.7834,0.0,0.15,0.122,0.0


## Verify
3 Gold tables should now exist: gold_inventory_health, gold_shipments_open,
gold_risk_scores. Check the risk_score column looks sane (0-100, HIGH/MEDIUM/
LOW bands) and that reliability_risk/rural_risk aren't all null (a null there
means a join upstream silently dropped rows -- tell me if you see that).